# Market Basket Analysis con Apriori

Esta actividad convierte cada vivienda en una transacción de atributos categóricos para encontrar asociaciones entre precio, área, habitaciones y baños.

**Pregunta central:** ¿las viviendas con precio alto tienden también a tener un área grande?\n
Importante: el dataset no registra compras. Por tanto, las reglas representan co-ocurrencias entre características de viviendas, no decisiones de compra ni causalidad.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
DATA_PATH = Path("data/houses.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/houses.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("No se encontró data/houses.csv desde la raíz o market_basket.")
df = pd.read_csv(DATA_PATH)
print(f"Viviendas analizadas: {len(df):,}")
display(df.head())

## 1. Convertir variables continuas en ítems

Apriori trabaja con presencia/ausencia de ítems. Por ello, precio y área se convierten en terciles: bajo, medio y alto. Los puntos de corte se calculan con los datos completos, incluidos los valores atípicos.

In [ ]:
price_q1, price_q2 = df["precio(mxn_peso)"].quantile([1/3, 2/3])
area_q1, area_q2 = df["area(m2)"].quantile([1/3, 2/3])
def price_category(value):
    if value <= price_q1:
        return "precio_bajo"
    if value <= price_q2:
        return "precio_medio"
    return "precio_alto"
def area_category(value):
    if value <= area_q1:
        return "area_pequena"
    if value <= area_q2:
        return "area_mediana"
    return "area_grande"
transactions = pd.DataFrame({
    "precio": df["precio(mxn_peso)"].map(price_category),
    "area": df["area(m2)"].map(area_category),
    "habitaciones": np.select(
        [df["habitaciones"] <= 2, df["habitaciones"] == 3],
        ["habitaciones_1_2", "habitaciones_3"],
        default="habitaciones_4_o_mas",
    ),
    "banos": np.select(
        [df["baños"] == 1, df["baños"] == 2],
        ["banos_1", "banos_2"],
        default="banos_3_o_mas",
    ),
})
print(f"Cortes de precio: ${price_q1:,.0f} y ${price_q2:,.0f}")
print(f"Cortes de área: {area_q1:.0f} m² y {area_q2:.0f} m²")
display(transactions.apply(pd.Series.value_counts).fillna(0).astype(int))

## 2. Matriz transaccional

Cada fila representa una vivienda y cada columna indica si el ítem está presente (`True`) o no (`False`). Una transacción contiene un ítem de cada grupo: precio, área, habitaciones y baños.

In [ ]:
items = sorted(set(transactions.to_numpy().ravel()))
basket = pd.DataFrame(False, index=transactions.index, columns=items)
for column in transactions.columns:
    basket.loc[transactions.index, transactions[column].unique()] = False
    for item in transactions[column].unique():
        basket.loc[transactions[column] == item, item] = True
print(f"Transacciones: {basket.shape[0]:,}")
print(f"Ítems: {basket.shape[1]}")
display(basket.head())

## 3. Algoritmo Apriori

Se utiliza `min_support = 0.05`: un conjunto de ítems debe aparecer al menos en el 5% de las viviendas para considerarse frecuente. Después se generan las reglas y se conservan las que tienen `confidence >= 0.50` y `lift > 1`.

In [ ]:
MIN_SUPPORT = 0.05
MIN_CONFIDENCE = 0.50
frequent_itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE,
)
rules = rules[rules["lift"] > 1].copy()
rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)
print(f"Conjuntos frecuentes encontrados: {len(frequent_itemsets)}")
print(f"Reglas positivas encontradas: {len(rules)}")
display(frequent_itemsets.sort_values(["length", "support"], ascending=[True, False]).head(15))

## 4. Reglas principales

La tabla muestra los ítems del antecedente y consecuente, además de soporte, confianza y lift. Una regla como `{area_grande} → {precio_alto}` significa que, dentro de esta muestra, se observa qué proporción de las viviendas grandes también pertenece al grupo de precio alto.

In [ ]:
def format_rule(value):
    return ", ".join(sorted(value))
rules_display = rules.assign(
    antecedente=rules["antecedents"].map(format_rule),
    consecuente=rules["consequents"].map(format_rule),
)[["antecedente", "consecuente", "support", "confidence", "lift"]]
display(rules_display.head(20).style.format({
    "support": "{:.3f}", "confidence": "{:.3f}", "lift": "{:.3f}"
}))

## 5. Asociación entre precio y área

Se filtran las reglas que conectan explícitamente una categoría de precio con una categoría de área. Esta es la parte que responde a la hipótesis inicial del análisis.

In [ ]:
price_area_rules = rules[
    rules["antecedents"].apply(lambda x: any(item.startswith("precio_") for item in x)) &
    rules["consequents"].apply(lambda x: any(item.startswith("area_") for item in x)) |
    rules["antecedents"].apply(lambda x: any(item.startswith("area_") for item in x)) &
    rules["consequents"].apply(lambda x: any(item.startswith("precio_") for item in x))
].copy()
price_area_display = price_area_rules.assign(
    antecedente=price_area_rules["antecedents"].map(format_rule),
    consecuente=price_area_rules["consequents"].map(format_rule),
)[["antecedente", "consecuente", "support", "confidence", "lift"]]
if price_area_display.empty:
    print("No se encontraron reglas precio-área con los umbrales actuales.")
else:
    display(price_area_display.style.format({
        "support": "{:.3f}", "confidence": "{:.3f}", "lift": "{:.3f}"
    }))

In [ ]:
if not price_area_rules.empty:
    plot_rules = price_area_rules.assign(
        regla=price_area_rules["antecedents"].map(format_rule) + " → " + price_area_rules["consequents"].map(format_rule)
    ).sort_values("lift", ascending=True).tail(12)
    plt.figure(figsize=(10, max(4, len(plot_rules) * 0.38)))
    sns.barplot(data=plot_rules, x="lift", y="regla", hue="lift", palette="viridis", legend=False)
    plt.axvline(1, color="red", linestyle="--", label="Lift = 1: independencia")
    plt.title("Reglas de asociación entre precio y área")
    plt.xlabel("Lift")
    plt.ylabel("")
    plt.legend()
    plt.tight_layout()
    plt.show()
plt.figure(figsize=(8, 5))
sns.scatterplot(data=rules, x="support", y="confidence", size="lift", hue="lift", palette="viridis", sizes=(40, 400), alpha=0.8)
plt.title("Soporte, confianza y lift de las reglas")
plt.xlabel("Soporte")
plt.ylabel("Confianza")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Conclusión

Las reglas con `lift > 1` muestran asociaciones positivas entre atributos. Para comunicar una regla se deben reportar siempre sus tres métricas. Por ejemplo, una confianza de 0.80 significa que el 80% de las viviendas con el antecedente también tiene el consecuente, mientras que el lift indica si esa proporción es mayor que la esperada por la frecuencia general del consecuente.

Estas asociaciones son descriptivas. No implican causalidad y no deben interpretarse como una probabilidad universal fuera de esta muestra.